In [1]:
%load_ext autoreload
%autoreload 2
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
cell_protein_df = pd.read_csv('../data/MIBI/raw/cell_protein_data.csv')
patient_df = pd.read_csv('../data/MIBI/raw/patient_info.csv')

In [3]:
import sys
import os
import numpy as np
from torch_geometric.loader import DataLoader

sys.path.append(os.path.abspath('../src'))
from utils.data_utils import split_dataset, split_indices
from dataset.placenta import PlacentaDatasetHypergraph
from dataset.extend import ExtendedDataset
from dataset.mibi import MIBIDataset, MIBISubsetHypergraph

def prepare_dataloaders(args):
    if args.dataset == 'mibi':
        dataset = MIBIDataset(data_folder=args.data_folder, k_hop=args.k_hop)

        ratios = [float(c) for c in args.train_val_test_ratio.split(':')]
        ratios = tuple([c / sum(ratios) for c in ratios])
        indices = list(range(len(dataset)))

        # NOTE: This is a hack to make sure all sets have all classes.
        all_class_subset1 = [0, 1, 4, 6]
        all_class_subset2 = [7, 8, 10, 11]
        all_class_subset3 = [12, 13, 14, 15]
        indices = list(set(indices) - set(all_class_subset1 + all_class_subset2 + all_class_subset3))
        adjusted_ratios = len(dataset) * np.array(ratios) - np.array([4, 4, 4])
        adjusted_ratios = tuple([c / sum(adjusted_ratios) for c in adjusted_ratios])
        train_indices, val_indices, test_indices = \
            split_indices(indices=indices, splits=adjusted_ratios, random_seed=0)
        train_indices += all_class_subset1
        val_indices += all_class_subset2
        test_indices += all_class_subset3

        train_set = MIBISubsetHypergraph(
            dataset=dataset,
            subset_indices=train_indices)
        val_set = MIBISubsetHypergraph(
            dataset=dataset,
            subset_indices=val_indices)
        test_set = MIBISubsetHypergraph(
            dataset=dataset,
            subset_indices=test_indices)

    min_batch_per_epoch = 5
    desired_len = args.desired_batch_size * min_batch_per_epoch
    if len(train_set) < desired_len:
        train_set = ExtendedDataset(dataset=train_set, desired_len=desired_len)

    train_loader = DataLoader(train_set, batch_size=args.batch_size, num_workers=args.num_workers, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=args.batch_size, num_workers=args.num_workers, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=args.batch_size, num_workers=args.num_workers, shuffle=False)

    return train_loader, val_loader, test_loader, dataset.num_classes

In [4]:
from argparse import Namespace

args = Namespace(
    dataset='mibi',
    data_folder='../data/MIBI/patchified_all_genes',
    train_val_test_ratio='6:2:2',
    desired_batch_size=16,
    batch_size=1,
    k_hop=1,
    num_workers=4,
    max_epochs=50,
    max_training_iters=512,
    max_validation_iters=256,
    learning_rate=1e-2,
    random_seed=1,
)

# Access with dot notation
print(args.dataset)  # 'MIBI'

mibi


In [5]:
dataset = MIBIDataset(data_folder=args.data_folder, k_hop=args.k_hop)
dataset

In [6]:
# Load the data.
train_loader, val_loader, test_loader, num_classes = prepare_dataloaders(args)

In [7]:
dataset = MIBIDataset(data_folder=args.data_folder, k_hop=args.k_hop)

ratios = [float(c) for c in args.train_val_test_ratio.split(':')]
ratios = tuple([c / sum(ratios) for c in ratios])
indices = list(range(len(dataset)))

# NOTE: This is a hack to make sure all sets have all classes.
all_class_subset1 = [0, 1, 4, 6]
all_class_subset2 = [7, 8, 10, 11]
all_class_subset3 = [12, 13, 14, 15]
indices = list(set(indices) - set(all_class_subset1 + all_class_subset2 + all_class_subset3))
adjusted_ratios = len(dataset) * np.array(ratios) - np.array([4, 4, 4])
adjusted_ratios = tuple([c / sum(adjusted_ratios) for c in adjusted_ratios])
train_indices, val_indices, test_indices = \
    split_indices(indices=indices, splits=adjusted_ratios, random_seed=0)
train_indices += all_class_subset1
val_indices += all_class_subset2
test_indices += all_class_subset3

In [8]:
dataset = MIBIDataset(data_folder=args.data_folder, k_hop=args.k_hop)

In [9]:
train_set = MIBISubsetHypergraph(
    dataset=dataset,
    subset_indices=train_indices)

In [10]:
from models.hypergraph_scattering import HypergraphScatteringNet
from utils.seed import seed_everything
from utils.log_utils import log
from utils.data_utils import split_dataset, split_indices
from utils.scheduler import LinearWarmupCosineAnnealingLR
from dataset.placenta import PlacentaDatasetHypergraph
from dataset.mibi import MIBIDataset, MIBISubsetHypergraph
from dataset.extend import ExtendedDataset
import torch

log_file = os.path.join('results', args.dataset, 'log.txt')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log(f'Using device: {device}', filepath=log_file, to_console=True)


Using device: cuda


In [11]:
log(f'Train set: {len(train_loader.dataset)}, Val set: {len(val_loader.dataset)}, Test set: {len(test_loader.dataset)}', filepath=log_file, to_console=True)

model = HypergraphScatteringNet(
    in_channels=64,
    hidden_channels=64,
    out_channels=num_classes,
    num_features=29,
    trainable_laziness=False,
    trainable_scales=True,
    activation=None,  # just get one layer of wavelet transform
    fixed_weights=True,
    layout=['hsm'],
    normalize='right',
    pooling='attention',
    scale_list=[0,1,2,4]
)
model.to(device)

# Check model parameters
total_params = 0
for param in model.parameters():
    total_params += param.numel() * param.element_size()
log(f"Parameters: {total_params/1e3:.2f}KB", filepath=log_file)

# Set up training tools.
optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate)
scheduler = LinearWarmupCosineAnnealingLR(
    optimizer,
    warmup_epochs=min(10, args.max_epochs),
    warmup_start_lr=args.learning_rate * 1e-2,
    max_epochs=args.max_epochs)
loss_fn = torch.nn.CrossEntropyLoss()

# Log the config.
config_str = 'Config: \n'
for key in vars(args).keys():
    config_str += '%s: %s\n' % (key, vars(args)[key])
config_str += '\nTraining History:'
log(config_str, filepath=log_file, to_console=True)

log(f'[HypergraphScattering] Training begins.', filepath=log_file)
best_val_auroc = 0
best_val_accuracy = 0

Train set: 3610, Val set: 1050, Test set: 1070
Parameters: 4.85KB
Config: 
dataset: mibi
data_folder: ../data/MIBI/patchified_all_genes
train_val_test_ratio: 6:2:2
desired_batch_size: 16
batch_size: 1
k_hop: 1
num_workers: 4
max_epochs: 50
max_training_iters: 512
max_validation_iters: 256
learning_rate: 0.01
random_seed: 1

Training History:
[HypergraphScattering] Training begins.


In [12]:
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score

def train_epoch(model, train_loader, optimizer, loss_fn, device, max_iter, num_classes):
    train_loss = 0
    y_true_arr, y_pred_arr = None, None
    optimizer.zero_grad()
    batch_per_backprop = int(args.desired_batch_size / args.batch_size)

    for iter_idx, data_item in enumerate(tqdm(train_loader)):
        if max_iter is not None and iter_idx > max_iter:
            break

        data_item = data_item.to(device)
        y_true = data_item.y.long()
        y_pred = model(
            x=data_item.x,
            hyperedge_index=data_item.edge_index,
            hyperedge_attr=data_item.edge_attr,
            batch=data_item.batch)

        loss = loss_fn(y_pred, y_true)

        loss_ = loss / batch_per_backprop
        loss_.backward()

        train_loss += loss.mean().item()

        # Simulate bigger batch size by batched optimizer update.
        if iter_idx % batch_per_backprop == batch_per_backprop - 1:
            optimizer.step()
            optimizer.zero_grad()

        y_true_np = y_true.detach().cpu().numpy()                        # shape: (batch size, 1)
        y_pred_np = torch.softmax(y_pred, dim=1).detach().cpu().numpy()  # shape: (batch size, num classes)
        if y_true_arr is None:
            y_true_arr = y_true_np
            y_pred_arr = y_pred_np
        else:
            y_true_arr = np.hstack((y_true_arr, y_true_np))
            y_pred_arr = np.vstack((y_pred_arr, y_pred_np))

    train_loss /= min(max_iter, len(train_loader))
    accuracy = accuracy_score(y_true_arr, np.argmax(y_pred_arr, axis=1))
    auroc = roc_auc_score(y_true_arr, y_pred_arr, multi_class='ovo', average='macro', labels=np.arange(num_classes))
    return model, train_loss, accuracy, auroc


In [13]:
@torch.no_grad()
def val_epoch(model, val_loader, loss_fn, device, max_iter, num_classes):
    val_loss = 0
    y_true_arr, y_pred_arr = None, None

    for iter_idx, data_item in enumerate(val_loader):
        if max_iter is not None and iter_idx > max_iter:
            break

        data_item = data_item.to(device)
        y_true = data_item.y.long()
        y_pred = model(
            x=data_item.x,
            hyperedge_index=data_item.edge_index,
            hyperedge_attr=data_item.edge_attr,
            batch=data_item.batch)
        loss = loss_fn(y_pred, y_true)

        val_loss += loss.mean().item()

        y_true_np = y_true.detach().cpu().numpy()                        # shape: (batch size, 1)
        y_pred_np = torch.softmax(y_pred, dim=1).detach().cpu().numpy()  # shape: (batch size, num classes)
        if y_true_arr is None:
            y_true_arr = y_true_np
            y_pred_arr = y_pred_np
        else:
            y_true_arr = np.hstack((y_true_arr, y_true_np))
            y_pred_arr = np.vstack((y_pred_arr, y_pred_np))

    val_loss /= min(max_iter, len(val_loader))
    accuracy = accuracy_score(y_true_arr, np.argmax(y_pred_arr, axis=1))
    # Check for invalid predictions
    print(y_pred_arr)
    print(np.arange(num_classes))
    print(f"y_pred has NaN: {np.isnan(y_pred_arr).any()}")
    print(f"y_pred has inf: {np.isinf(y_pred_arr).any()}")
    print(f"y_pred range: [{y_pred_arr.min():.3f}, {y_pred_arr.max():.3f}]")
    auroc = roc_auc_score(y_true_arr, y_pred_arr, multi_class='ovo', average='macro', labels=np.arange(num_classes))
    return model, val_loss, accuracy, auroc

In [16]:
model_save_path = os.path.join('results','mibi','model.pt')

In [18]:
for epoch_idx in tqdm(range(args.max_epochs)):
    model.train()
    model, train_loss, train_accuracy, train_auroc = train_epoch(model, train_loader, optimizer, loss_fn, device, args.max_training_iters, num_classes)
    scheduler.step()
    log_lr = optimizer.param_groups[0]['lr']
    log(f'Epoch {epoch_idx + 1}/{args.max_epochs}: (LR={log_lr}) Training Loss {train_loss:.3f}, ACC {train_accuracy:.3f}, macro AUROC {train_auroc:.3f}.',
        filepath=log_file)

    model.eval()
    model, val_loss, val_accuracy, val_auroc = val_epoch(model, val_loader, loss_fn, device, args.max_validation_iters, num_classes)
    log(f'Validation Loss {val_loss:.3f}, ACC {val_accuracy:.3f}, macro AUROC {val_auroc:.3f}.',
        filepath=log_file)
    # if val_auroc > best_val_auroc:
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        torch.save(model.state_dict(), model_save_path)
        log('Model weights successfully saved.', filepath=log_file)

 14%|█▍        | 513/3610 [00:55<05:36,  9.20it/s]

Epoch 1/50: (LR=0.005600000000000001) Training Loss 1.258, ACC 0.441, macro AUROC 0.676.



  2%|▏         | 1/50 [01:22<1:07:33, 82.73s/it]

[[0.2342301  0.6399408  0.09494728 0.03088188]
 [0.37279084 0.47620937 0.11445735 0.03654244]
 [0.58260626 0.12779959 0.20466143 0.08493274]
 ...
 [0.5047646  0.27191922 0.17846957 0.04484661]
 [0.52934676 0.22975011 0.18034469 0.06055846]
 [0.2886032  0.36229628 0.23099428 0.11810622]]
[0 1 2 3]
y_pred has NaN: False
y_pred has inf: False
y_pred range: [0.003, 0.953]
Validation Loss 1.532, ACC 0.331, macro AUROC 0.396.


 14%|█▍        | 513/3610 [00:53<05:24,  9.55it/s]

...

[[3.4028119e-07 9.9777287e-01 1.0397297e-09 2.2267525e-03]
 [8.0383480e-02 5.1524973e-01 3.7442738e-01 2.9939419e-02]
 [7.8956194e-02 5.2296516e-02 8.6693347e-01 1.8138011e-03]
 ...
 [9.8169506e-01 1.4162072e-04 1.8163260e-02 4.6080997e-10]
 [4.8366910e-01 1.5898387e-01 3.5734344e-01 3.6765066e-06]
 [2.1436041e-02 4.6226475e-01 5.1626462e-01 3.4588738e-05]]
[0 1 2 3]
y_pred has NaN: False
y_pred has inf: False
y_pred range: [0.000, 1.000]
Validation Loss 4.289, ACC 0.218, macro AUROC 0.624.


 14%|█▍        | 513/3610 [00:55<05:37,  9.17it/s]

Epoch 50/50: (LR=0.00024471741852424004) Training Loss 0.057, ACC 0.981, macro AUROC 1.000.



100%|██████████| 50/50 [1:10:43<00:00, 84.87s/it]

[[3.1156111e-07 9.9779189e-01 1.0144614e-09 2.2077027e-03]
 [8.3173156e-02 4.9851626e-01 3.8954747e-01 2.8763136e-02]
 [7.8959055e-02 4.3786455e-02 8.7574673e-01 1.5077017e-03]
 ...
 [9.8407817e-01 1.1838051e-04 1.5803520e-02 3.1928640e-10]
 [4.9818265e-01 1.4494783e-01 3.5686663e-01 2.8993197e-06]
 [2.0642577e-02 4.2216900e-01 5.5715775e-01 3.0691972e-05]]
[0 1 2 3]
y_pred has NaN: False
y_pred has inf: False
y_pred range: [0.000, 1.000]
Validation Loss 4.328, ACC 0.218, macro AUROC 0.626.
